# Day 8 · Exercise 4: Multi-Label Tagger

**What you'll build:** `classify_multi_label(text: str, labels: list[str], model: str) -> list[str]` — a function that prompts a local Ollama model to return every applicable label from an allowed set, then parses, validates, and deduplicates the result into a clean Python list.

**Why it matters:** Real-world text rarely fits one category — a support ticket can be about billing *and* a technical error at the same time; being able to assign multiple labels programmatically is the foundation of any serious text-routing or tagging pipeline.

## Your Implementation

In [ ]:
import json
import ollama


def classify_multi_label(text: str, labels: list[str], model: str) -> list[str]:
    """Return all labels from `labels` that apply to `text`.

    Prompts the Ollama model to evaluate every label in `labels` independently
    and return a JSON array of those that apply.  The raw reply is parsed with
    json.loads first; if that fails a comma-separated fallback is tried.  Every
    candidate is then validated against the allowed set (case-insensitive) and
    deduplicated before being returned.

    Args:
        text:   The input text to tag.
        labels: The full set of allowed label strings.
        model:  Ollama model name (e.g. "llama3.2").

    Returns:
        A (possibly empty) list of labels drawn from `labels` that apply to
        `text`.  No duplicates; canonical spelling from `labels` is preserved.

    Example:
        >>> result = classify_multi_label(
        ...     "I was charged twice and can't log in.",
        ...     ["billing", "account-access", "shipping"],
        ...     "llama3.2",
        ... )
        >>> set(result) == {"billing", "account-access"}
        True
    """
    # ── YOUR CODE HERE ─────────────────────────────────────────
    pass
    # ───────────────────────────────────────────────────────────

## Check Your Work

Run the cell below — it runs 5 automated checks and shows ✅ / ❌ for each.

In [ ]:
_PASS, _FAIL = '✅', '❌'

_ALLOWED = [
    "billing",
    "technical-issue",
    "feature-request",
    "shipping",
    "account-access",
]


def _run_checks():
    score, total = 0, 6

    # Check 1: function exists and is callable
    try:
        assert callable(classify_multi_label), 'classify_multi_label is not defined'
        print(f'{_PASS} Check 1/{total}: function exists and is callable')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 1/{total}: {e}')
        return  # Cannot safely run further checks

    # Check 2: always returns a list
    try:
        result = classify_multi_label(
            "Thank you for the great service!", _ALLOWED, "llama3.2"
        )
        assert isinstance(result, list), f'expected list, got {type(result).__name__}'
        print(f'{_PASS} Check 2/{total}: return type is list')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 2/{total}: {e}')
        return  # No point checking contents if type is wrong

    # Check 3: every returned item is in the allowed set
    try:
        test_text = "My package has been sitting in the warehouse for two weeks."
        result = classify_multi_label(test_text, _ALLOWED, "llama3.2")
        rogue = [item for item in result if item not in _ALLOWED]
        assert not rogue, f'labels not in allowed set: {rogue}'
        print(f'{_PASS} Check 3/{total}: all returned labels are in the allowed set')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 3/{total}: {e}')

    # Check 4: no duplicates in the returned list
    try:
        test_text = "I was charged twice and my card was billed for the wrong amount."
        result = classify_multi_label(test_text, _ALLOWED, "llama3.2")
        assert len(result) == len(set(result)), f'duplicates found in result: {result}'
        print(f'{_PASS} Check 4/{total}: no duplicates in the returned list')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 4/{total}: {e}')

    # Check 5: clearly off-topic text returns an empty list
    try:
        off_topic = "Happy birthday! Hope you have a wonderful day."
        result = classify_multi_label(off_topic, _ALLOWED, "llama3.2")
        assert isinstance(result, list), f'expected list, got {type(result).__name__}'
        assert result == [], (
            f'expected [] for off-topic text, got {result!r} — '
            'make sure your prompt explicitly allows an empty array'
        )
        print(f'{_PASS} Check 5/{total}: off-topic text returns an empty list')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 5/{total}: {e}')

    # Check 6: text covering two topics returns at least two labels
    try:
        two_topic = "I was charged twice and now I cannot log into my account."
        result = classify_multi_label(two_topic, _ALLOWED, "llama3.2")
        assert isinstance(result, list), f'expected list, got {type(result).__name__}'
        assert len(result) >= 2, (
            f'expected at least 2 labels for a text about billing AND account access, '
            f'got {result!r} — check that your prompt asks for ALL applicable labels'
        )
        print(f'{_PASS} Check 6/{total}: two-topic text returns at least two labels (got {result})')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 6/{total}: {e}')

    print()
    if score == total:
        print('🎉 Exercise complete!')
        print(f'  {score}/{total} passed.')
    else:
        print(f'  {score}/{total} passed. Keep going!')


_run_checks()

## Bonus Challenge

Add a `threshold` parameter (a float between 0 and 1) and ask the model to rate
each label's confidence score alongside the label name, returning a JSON array of
objects like `[{"label": "billing", "score": 0.95}]`.  Filter out any label whose
score falls below `threshold` before returning the list.

This foreshadows Day 9's work on structured JSON output — where you will use
schema validation to make exactly this kind of rich structured reply reliable.

## Solution

<details>
<summary>Click to reveal — try on your own first</summary>

```python
import json
import ollama


MULTI_LABEL_SYSTEM_PROMPT = """You are a text tagger.

Allowed labels:
{labels}

Task:
- Read the text and decide which of the allowed labels apply to it.
- A label applies only if it clearly and specifically describes content in the text.
- Return ALL applicable labels as a JSON array of strings.
- If no label applies, return an empty array: []
- Output only the JSON array. No explanation. No punctuation outside the array.

Example output for two matching labels:
["billing", "technical-issue"]"""


def _validate_labels(candidates: list, allowed: list[str]) -> list[str]:
    """Keep only valid labels; deduplicate; preserve canonical spelling."""
    allowed_lower = {lb.lower(): lb for lb in allowed}
    seen: set[str] = set()
    result: list[str] = []
    for item in candidates:
        if not isinstance(item, str):
            continue
        key = item.strip().lower()
        if key in allowed_lower and key not in seen:
            seen.add(key)
            result.append(allowed_lower[key])
    return result


def _parse_label_list(raw: str, allowed: list[str]) -> list[str]:
    """Parse the model's reply into a validated, deduplicated label list."""
    # Attempt 1 — JSON array
    try:
        parsed = json.loads(raw)
        if isinstance(parsed, list):
            return _validate_labels(parsed, allowed)
    except json.JSONDecodeError:
        pass

    # Attempt 2 — CSV fallback
    candidates = [item.strip() for item in raw.split(",") if item.strip()]
    return _validate_labels(candidates, allowed)


def classify_multi_label(text: str, labels: list[str], model: str) -> list[str]:
    """Return all labels from `labels` that apply to `text`.

    Prompts the Ollama model to evaluate every label in `labels` independently
    and return a JSON array of those that apply.  The raw reply is parsed with
    json.loads first; if that fails a comma-separated fallback is tried.  Every
    candidate is then validated against the allowed set (case-insensitive) and
    deduplicated before being returned.

    Args:
        text:   The input text to tag.
        labels: The full set of allowed label strings.
        model:  Ollama model name (e.g. "llama3.2").

    Returns:
        A (possibly empty) list of labels drawn from `labels` that apply to
        `text`.  No duplicates; canonical spelling from `labels` is preserved.

    Example:
        >>> result = classify_multi_label(
        ...     "I was charged twice and can't log in.",
        ...     ["billing", "account-access", "shipping"],
        ...     "llama3.2",
        ... )
        >>> set(result) == {"billing", "account-access"}
        True
    """
    label_str = ", ".join(labels)
    system_msg = MULTI_LABEL_SYSTEM_PROMPT.format(labels=label_str)

    messages = [
        {"role": "system", "content": system_msg},
        {
            "role": "user",
            "content": (
                f"Text:\n{text}\n\n"
                f"Allowed labels: {label_str}\n"
                "Reply with a JSON array of applicable labels only."
            ),
        },
    ]

    response = ollama.chat(model=model, messages=messages)
    raw: str = response["message"]["content"].strip()
    return _parse_label_list(raw, labels)
```

**Why this works:** The system prompt explicitly asks for a JSON array *and* gives a concrete example output, which anchors the model to the right format far more reliably than a prose description alone.  `_parse_label_list` tries `json.loads` first — the fast, exact path — and only falls back to comma-splitting if the model strays from valid JSON, making the function robust to both reply styles.  `_validate_labels` then applies a case-insensitive lookup against the allowed set and tracks already-seen labels with a `set`, so the final list is always clean, canonical, and duplicate-free regardless of what the model returned.
</details>